# WeatherForYnov — Modélisation prédictive

**Objectif** : prédire la **température moyenne mensuelle** (`temp_moy_mensuelle`) par **département**, à partir de 4 semaines de données passées (28 jours) et de features climatiques.

> Exécuter d'abord `01_exploration_donnees.ipynb` pour générer les fichiers dans `src/data/processed/`.

## 1. Configuration

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED_DIR = ROOT / "src" / "data" / "processed"
DEPT_PATH = PROCESSED_DIR / "climat_mensuel_departements.parquet"
if not DEPT_PATH.exists():
    DEPT_PATH = PROCESSED_DIR / "climat_mensuel_departements.csv"

df = pd.read_parquet(DEPT_PATH) if DEPT_PATH.suffix == ".parquet" else pd.read_csv(DEPT_PATH, parse_dates=["date"])
df = df.sort_values(["code_departement", "date"]).reset_index(drop=True)
print(f"Dataset : {len(df):,} lignes, {df['code_departement'].nunique()} départements")

## 2. Feature engineering — Lags (4 semaines ≈ 1 mois précédent)

Comme les données sont **mensuelles**, on utilise les **3 mois précédents** comme proxy des 4 semaines de contexte, plus les features du mois courant (sans la cible).

In [ ]:
FEATURES_BASE = [
    "temp_max", "temp_min", "precipitations_mm",
    "humidite", "vent_moyen", "insolation",
]
FEATURES_BASE = [c for c in FEATURES_BASE if c in df.columns]
TARGET = "temp_moy_mensuelle"

LAG_MONTHS = [1, 2, 3]

for col in FEATURES_BASE:
    for lag in LAG_MONTHS:
        df[f"{col}_lag{lag}"] = df.groupby("code_departement")[col].shift(lag)

# Features calendaires
df["mois_sin"] = np.sin(2 * np.pi * df["mois"] / 12)
df["mois_cos"] = np.cos(2 * np.pi * df["mois"] / 12)

lag_cols = [f"{c}_lag{l}" for c in FEATURES_BASE for l in LAG_MONTHS]
feature_cols = FEATURES_BASE + lag_cols + ["mois", "trimestre", "trend_index", "mois_sin", "mois_cos"]
feature_cols = [c for c in feature_cols if c in df.columns]

ml = df.dropna(subset=feature_cols + [TARGET]).copy()
print(f"Lignes ML après lags : {len(ml):,}")
ml.head()

## 3. Split temporel (train / test)

In [ ]:
CUTOFF_YEAR = 2018  # Train avant 2018, test à partir de 2018

train = ml[ml["annee"] < CUTOFF_YEAR]
test = ml[ml["annee"] >= CUTOFF_YEAR]

X_train, y_train = train[feature_cols], train[TARGET]
X_test, y_test = test[feature_cols], test[TARGET]

print(f"Train : {len(train):,} | Test : {len(test):,}")

## 4. Entraînement et évaluation

In [ ]:
models = {
    "Ridge": Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))]),
    "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    results.append({
        "modele": name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R2": r2_score(y_test, y_pred),
    })

results_df = pd.DataFrame(results).sort_values("MAE")
results_df

## 5. Visualisation des prédictions

In [ ]:
best_model = results_df.iloc[0]["modele"]
y_pred_best = predictions[best_model]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred_best, alpha=0.2, s=8)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[0].set_xlabel("Réel (°C)")
axes[0].set_ylabel("Prédit (°C)")
axes[0].set_title(f"Prédictions vs Réel — {best_model}")

residuals = y_test.values - y_pred_best
sns.histplot(residuals, bins=40, kde=True, ax=axes[1])
axes[1].set_title("Distribution des résidus")
axes[1].set_xlabel("Erreur (°C)")
plt.tight_layout()
plt.show()

## 6. Évaluation par région

In [ ]:
eval_region = test[["nom_region", TARGET]].copy()
eval_region["prediction"] = y_pred_best
eval_region["erreur_abs"] = (eval_region[TARGET] - eval_region["prediction"]).abs()

region_mae = (
    eval_region.groupby("nom_region")["erreur_abs"]
    .mean()
    .sort_values()
    .round(3)
    .rename("MAE_region")
)
region_mae

## 7. Importance des features (Random Forest)

In [ ]:
rf = models["RandomForest"]
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
importances.head(15).plot(kind="barh", ax=ax)
ax.set_title("Top 15 — Importance des features (Random Forest)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Prochaines améliorations

- Passer aux **données journalières** pour respecter exactement la contrainte « 4 semaines de données passées »
- Ajouter des features externes : **éruptions solaires**, Open-Meteo, couverture nuageuse
- Tester des modèles temporels : **Prophet**, **XGBoost**, **LSTM**
- Validation croisée temporelle par département
- Export du modèle pour l'application Dash/Plotly